In [1]:
from dotenv import load_dotenv
import os

from langchain_community.graphs import Neo4jGraph

# Warning control
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Load from environment
load_dotenv('.env', override=True)
NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

# Note the code below is unique to this course environment, and not a 
# standard part of Neo4j's integration with OpenAI. Remove if running 
# in your own environment.
#OPENAI_ENDPOINT = os.getenv('OPENAI_BASE_URL') + '/embeddings'

In [7]:
# OpenAI client
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

def get_embedding(text):
    return client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    ).data[0].embedding

In [9]:
#Create Neo4j Connection
kg = Neo4jGraph(
    url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE
)

In [4]:
#Create a vector index
kg.query("""
  CREATE VECTOR INDEX movie_tagline_embeddings IF NOT EXISTS
  FOR (m:Movie) ON (m.taglineEmbedding) 
  OPTIONS { indexConfig: {
    `vector.dimensions`: 1536,
    `vector.similarity_function`: 'cosine'
  }}"""
)

[]

In [5]:
kg.query("""
  SHOW VECTOR INDEXES
  """
)

[{'id': 3,
  'name': 'movie_tagline_embeddings',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'VECTOR',
  'entityType': 'NODE',
  'labelsOrTypes': ['Movie'],
  'properties': ['taglineEmbedding'],
  'indexProvider': 'vector-3.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0}]

In [11]:
#Populate the vector index
# Step 1: Fetch movies
movies = kg.query("""
MATCH (m:Movie)
WHERE m.tagline IS NOT NULL
RETURN elementId(m) AS id, m.tagline AS tagline
""")

In [13]:
# Step 2: Generate embeddings + store
for movie in movies:
    embedding = get_embedding(movie["tagline"])
    
    kg.query("""
    MATCH (m:Movie)
    WHERE elementId(m) = $id
    SET m.taglineEmbedding = $embedding
    """, params={
        "id": movie["id"],
        "embedding": embedding
    })

In [14]:
# =========================================
# Verify embeddings
# =========================================
result = kg.query("""
    MATCH (m:Movie) 
    WHERE m.tagline IS NOT NULL
    RETURN m.tagline, m.taglineEmbedding
    LIMIT 1
""")

print(result[0]['m.tagline'])
print(result[0]['m.taglineEmbedding'][:10])
print(len(result[0]['m.taglineEmbedding']))

Welcome to the Real World
[0.043975830078125, -0.0010557174682617188, 0.00748443603515625, 0.07147216796875, 0.0157470703125, -0.019287109375, -0.0269317626953125, 0.0169677734375, -0.035797119140625, 0.0279998779296875]
1536


In [15]:
# =========================================
# Similarity search (FIXED)
# =========================================
question = "What movies are about love?"
question_embedding = get_embedding(question)

kg.query("""
CALL db.index.vector.queryNodes(
    'movie_tagline_embeddings', 
    $top_k, 
    $embedding
) YIELD node AS movie, score
RETURN movie.title, movie.tagline, score
""", params={
    "embedding": question_embedding,
    "top_k": 5
})

[{'movie.title': 'Joe Versus the Volcano',
  'movie.tagline': 'A story of love, lava and burning desire.',
  'score': 0.6820023059844971},
 {'movie.title': 'Snow Falling on Cedars',
  'movie.tagline': 'First loves last. Forever.',
  'score': 0.6721906661987305},
 {'movie.title': 'When Harry Met Sally',
  'movie.tagline': 'Can two friends sleep together and still love each other in the morning?',
  'score': 0.6441078186035156},
 {'movie.title': 'Twister',
  'movie.tagline': "Don't Breathe. Don't Look Back.",
  'score': 0.6388194561004639},
 {'movie.title': "You've Got Mail",
  'movie.tagline': 'At odds in life... in love on-line.',
  'score': 0.6268529891967773}]

In [16]:
# =========================================
# Try for yourself: ask your own question!
# =========================================
question = "What movies are about adventure?"
question_embedding = get_embedding(question)

kg.query("""
CALL db.index.vector.queryNodes(
    'movie_tagline_embeddings', 
    $top_k, 
    $embedding
) YIELD node AS movie, score
RETURN movie.title, movie.tagline, score
""", params={
    "embedding": question_embedding,
    "top_k": 5
})

[{'movie.title': 'Twister',
  'movie.tagline': "Don't Breathe. Don't Look Back.",
  'score': 0.6612944602966309},
 {'movie.title': 'RescueDawn',
  'movie.tagline': "Based on the extraordinary true story of one man's fight for freedom",
  'score': 0.6467363834381104},
 {'movie.title': 'Joe Versus the Volcano',
  'movie.tagline': 'A story of love, lava and burning desire.',
  'score': 0.6254544258117676},
 {'movie.title': 'Cast Away',
  'movie.tagline': 'At the edge of the world, his journey begins.',
  'score': 0.6237642765045166},
 {'movie.title': 'Bicentennial Man',
  'movie.tagline': "One robot's 200 year journey to become an ordinary man.",
  'score': 0.6095681190490723}]